In [0]:
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
volume_path = dbutils.widgets.get("volume_path")
filename_format = dbutils.widgets.get("filename_format")

In [0]:
from datetime import datetime, timedelta

def list_dates_in_range(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    dates = [(start + timedelta(days=i)).strftime("%Y-%m-%d") for i in range((end - start).days + 1)]
    return dates

date_list = list_dates_in_range(start_date, end_date)

In [0]:
file_list=[]
for fetch_date in date_list:
    if "MMDDYY" in filename_format:
        file_date = fetch_date[5:7] + fetch_date[8:10] + fetch_date[2:4]
        file_name = filename_format.replace("MMDDYY", file_date)
    elif "YYYYMMDD" in filename_format:
        file_date = fetch_date[0:4] + fetch_date[5:7] + fetch_date[8:10]
        file_name = filename_format.replace("YYYYMMDD", file_date)
    else: 
        raise ValueError("The file format is not correct. Expected 'MMDDYY' or 'YYYYMMDD' in filename_format.")
    file_list.append(file_name)

In [0]:
not_found_files=[]
for file_name in file_list:
    try:
        dbutils.fs.ls(volume_path+file_name)
    except Exception as e:    
        not_found_files.append(file_name)
not_found_files

In [0]:
files_found = [f for f in file_list if f not in not_found_files]
if files_found == []:
    dbutils.jobs.taskValues.set(key="file_found", value="NOT-FOUND")
else:    
    dbutils.jobs.taskValues.set(key="file_found", value="FOUND")
    dbutils.jobs.taskValues.set(key="file_list", value = files_found)